# Demo 02 : Subgraph communication

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


In [2]:
import json
import operator
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.calls = self.input_tokens = self.output_tokens = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for batch in response.generations:
                for generation in batch:
                    usage = generation.message.usage_metadata or {}
                    self.input_tokens += usage.get("input_tokens", 0)
                    self.output_tokens += usage.get("output_tokens", 0)


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=500, max_retries=0)
started = time.perf_counter()
DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
NOTEBOOK_DIR = ROOT / "notebooks" / "06-agent-communication" if ROOT else Path.cwd()


def stock(part="CE-CF-02"):
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        db.row_factory = sqlite3.Row
        row = db.execute("SELECT part_no, name, stock, price_eur, depot FROM parts WHERE part_no = ?", (part,)).fetchone()
    return dict(row) if row else {"part_no": part, "error": "unknown part"}


def guidance(error="F-CE-110"):
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error + " |"))


FACTS = stock()
QUESTION = "Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110."
LIMIT = {"recursion_limit": 16}


def finish_metrics(name):
    row = {"model": MODEL, "rps": float(os.environ.get("COURSE_RPS", "2")),
           "calls": counter.calls, "input_tokens": counter.input_tokens,
           "output_tokens": counter.output_tokens, "seconds": round(time.perf_counter() - started, 1)}
    print("metrics :", row)
    (NOTEBOOK_DIR / (name + "-metrics.json")).write_text(json.dumps(row, indent=2) + "\n", encoding="utf-8")

## a · handoff-to-parent

We transfer from a parts subgraph to a manuals node in its parent. Code fixes the route so we can inspect the communication boundary without relying on a model's transfer choice.

In [3]:
def parts_reply(state):
    reply = llm.invoke([SystemMessage("Return the part count and depot briefly.\n" + json.dumps(FACTS)),
                        *state["messages"]])
    return {"messages": [reply]}


def handoff(state):
    return Command(goto="manuals", graph=Command.PARENT,
                   update={"messages": state["messages"]})


child = StateGraph(MessagesState)
child.add_node("parts_reply", parts_reply)
child.add_node("handoff", handoff)
child.add_edge(START, "parts_reply")
child.add_edge("parts_reply", "handoff")


def manuals_reply(state):
    print("receiver :", [(m.type, m.content) for m in state["messages"]])
    return {"messages": [llm.invoke([SystemMessage("Combine the visible stock and manual facts.\n" + guidance()),
                                    *state["messages"]])]}


parent = StateGraph(MessagesState)
parent.add_node("parts", child.compile())
parent.add_node("manuals", manuals_reply)
parent.add_edge(START, "parts")
parent.add_edge("manuals", END)
result = parent.compile().invoke({"messages": [HumanMessage(QUESTION, id="case-request")]}, LIMIT)
print("answer :", result["messages"][-1].content)
print("message ids :", [m.id for m in result["messages"]])

receiver : [('human', 'Report CE-CF-02 stock and location, then the first action and replacement condition for F-CE-110.'), ('ai', 'CE-CF-02: stock 6, at Rietpoort. I don’t have the first action or replacement condition for F-CE-110.')]


answer : CE-CF-02: I don’t have its stock or location in the available information.

F-CE-110: First, check the traction cooling fan. Replace it if airflow is below 80 percent.
message ids : ['case-request', 'lc_run--01a17a3f-4b5e-7050-bf17-904dd6501d04-0', 'lc_run--01a17a3f-5aaf-7bd1-b9dd-a22319f30372-0']


The parent's add_messages reducer merges the forwarded history by message id. The receiver's printed list includes the stock note. The written answer can still omit the stock count. A return annotation of Command with a literal destination would register that name as an edge inside the child, which has no such node. A tool-triggered handoff also needs a matching ToolMessage receipt and one tool call at a time, as module 05 showed. This node-triggered transfer has no pending tool call.

## b · shared-keys

We compile the parts graph as a subgraph. It shares part_no and stock with its parent. Its working note remains inside the child schema.

In [4]:
class PartsState(TypedDict):
    part_no: str
    stock: dict
    working_note: str


class CaseState(TypedDict):
    case_id: str
    part_no: str
    stock: dict
    answer: str


parts = StateGraph(PartsState)
parts.add_node("lookup", lambda s: {"stock": stock(s["part_no"]), "working_note": "read-only inventory"})
parts.add_edge(START, "lookup")
parts.add_edge("lookup", END)
parts_graph = parts.compile()
case = StateGraph(CaseState)
case.add_node("parts", parts_graph)
case.add_node("report", lambda s: {"answer": llm.invoke("Report these facts briefly.\n" + json.dumps(s["stock"])).content})
case.add_edge(START, "parts")
case.add_edge("parts", "report")
case.add_edge("report", END)
result = case.compile().invoke({"case_id": "BW-7001", "part_no": "CE-CF-02"})
print("parent output :", result)
print("private key returned :", "working_note" in result)

parent output : {'case_id': 'BW-7001', 'part_no': 'CE-CF-02', 'stock': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}, 'answer': 'Traction cooling fan (CE-CF-02): 6 in stock at Rietpoort, priced at €420 each.'}
private key returned : False


## c · transformed-input-and-output

We map a parent item field into the child's part_no field and return a selected stock finding. The child receives no parent-only requester field.

In [5]:
class RequestState(TypedDict):
    item: str
    requester: str
    finding: dict
    answer: str


def mapped_parts(state):
    child_input = {"part_no": state["item"]}
    print("child input :", child_input)
    child_output = parts_graph.invoke(child_input)
    print("child output :", child_output)
    return {"finding": child_output["stock"]}


mapped = StateGraph(RequestState)
mapped.add_node("parts", mapped_parts)
mapped.add_node("report", lambda s: {"answer": llm.invoke("Write a stock note from these facts.\n" + json.dumps(s["finding"])).content})
mapped.add_edge(START, "parts")
mapped.add_edge("parts", "report")
mapped.add_edge("report", END)
print("mapped parent :", mapped.compile().invoke({"item": "CE-CF-02", "requester": "workshop planner"}))

child input : {'part_no': 'CE-CF-02'}
child output : {'part_no': 'CE-CF-02', 'stock': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}, 'working_note': 'read-only inventory'}


mapped parent : {'item': 'CE-CF-02', 'requester': 'workshop planner', 'finding': {'part_no': 'CE-CF-02', 'name': 'traction cooling fan', 'stock': 6, 'price_eur': 420.0, 'depot': 'Rietpoort'}, 'answer': '**Stock note — CE-CF-02**\n\nTraction cooling fan stock at the Rietpoort depot: **6 units**. Price: **€420.00 each**.'}


We treat the wrapper as a data interface. Excluding a field from model input does not isolate credentials, Python memory, checkpoints or logs.

## Things to try

Optional : we can change the part, the context boundary or the policy and inspect the result.

In [6]:
finish_metrics("demo02")

metrics : {'model': 'openai/gpt-6-luna', 'rps': 0.4, 'calls': 4, 'input_tokens': 316, 'output_tokens': 154, 'seconds': 11.6}
